<a href="https://colab.research.google.com/github/Metropoliya/AI/blob/main/Voice%20Video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# ШАГ 1: ПОДКЛЮЧАЕМ GOOGLE ДИСК
# ==========================================
from google.colab import drive
import os
import shutil

print("Подключаем Google Диск для надежного сохранения...")
drive.mount('/content/drive')

# Создаем структуру папок прямо на твоем Диске
WORK_DIR = "/content/drive/MyDrive/Voice_Clone_Project"
CHUNKS_DIR = f"{WORK_DIR}/chunks"
os.makedirs(CHUNKS_DIR, exist_ok=True)

# Ищем твой файл. Если он загружен во временную память, копируем его на Диск
SOURCE_TEMP = "/content/026-03-18_23-54-17.mp3"
SOURCE_DRIVE = f"{WORK_DIR}/026-03-18_23-54-17.mp3"

if os.path.exists(SOURCE_TEMP) and not os.path.exists(SOURCE_DRIVE):
    print("Копируем исходный аудиофайл на Google Диск...")
    shutil.copy(SOURCE_TEMP, SOURCE_DRIVE)
elif not os.path.exists(SOURCE_DRIVE):
    print(f"ВНИМАНИЕ: Загрузи файл 026-03-18_23-54-17.mp3 в папку {WORK_DIR} на твоем Гугл Диске!")

# ==========================================
# ШАГ 2: ПОДГОТОВКА ИЗОЛИРОВАННОЙ СРЕДЫ (CONDA + GPU)
# ==========================================
if not os.path.exists('/usr/local/bin/conda'):
    print("Устанавливаем Conda...")
    !wget -qO miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh > /dev/null
    !bash miniconda.sh -b -f -p /usr/local > /dev/null
    !/usr/local/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main > /dev/null 2>&1
    !/usr/local/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r > /dev/null 2>&1

if not os.path.exists('/usr/local/envs/clone_env'):
    print("Создаем среду Python 3.10...")
    !/usr/local/bin/conda create -n clone_env python=3.10 -y -q > /dev/null

print("Устанавливаем правильные версии библиотек (с поддержкой видеокарты)...")
!apt-get install -y ffmpeg espeak-ng > /dev/null 2>&1
!/usr/local/bin/conda run -n clone_env pip install -q TTS openai-whisper pydub soundfile transformers==4.36.2
!/usr/local/bin/conda run -n clone_env pip install -q torch==2.1.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121 --upgrade

# ==========================================
# ШАГ 3: ЖЕЛЕЗОБЕТОННЫЙ СКРИПТ (РАБОТАЕТ НАПРЯМУЮ С ДИСКОМ)
# ==========================================
code = f"""
import os
import json
os.environ["COQUI_TOS_AGREED"] = "1"
import whisper
import torch
from TTS.api import TTS
from pydub import AudioSegment

WORK_DIR = "{WORK_DIR}"
CHUNKS_DIR = "{CHUNKS_DIR}"
my_file = "{SOURCE_DRIVE}"
segments_file = os.path.join(WORK_DIR, "segments.json")
speaker_sample = os.path.join(WORK_DIR, "speaker_sample.wav")

print(f"\\n>>> ПРОВЕРКА GPU: Видеокарта доступна? {{torch.cuda.is_available()}} <<<\\n")

print("--- ЭТАП 1: Подготовка эталона ---")
if not os.path.exists(speaker_sample):
    audio = AudioSegment.from_mp3(my_file)
    audio[10000:20000].export(speaker_sample, format="wav")
    print("Эталон голоса сохранен на Диск.")

print("--- ЭТАП 2: Перевод ---")
if os.path.exists(segments_file):
    print("Найден сохраненный перевод на Диске! Загружаем...")
    with open(segments_file, "r", encoding="utf-8") as f:
        segments = json.load(f)
else:
    print("Переводим аудио в текст...")
    model_whisper = whisper.load_model("small")
    result = model_whisper.transcribe(my_file, task="translate", condition_on_previous_text=False)
    segments = result["segments"]
    with open(segments_file, "w", encoding="utf-8") as f:
        json.dump(segments, f, ensure_ascii=False, indent=4)
    del model_whisper
    torch.cuda.empty_cache()

print(f"Всего фраз для озвучки: {{len(segments)}}")

print("--- ЭТАП 3: Озвучка с сохранением на Google Диск ---")
device = "cuda" if torch.cuda.is_available() else "cpu"
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

final_audio = AudioSegment.empty()

for i, segment in enumerate(segments):
    text = segment["text"].strip()
    if not text:
        continue

    out_path = os.path.join(CHUNKS_DIR, f"chunk_{{i}}.wav")

    # ЖЕСТКАЯ ПРОВЕРКА: Если файл уже есть на Диске, размер > 0, пропускаем генерацию
    if os.path.exists(out_path) and os.path.getsize(out_path) > 100:
        print(f"Фраза [{{i+1}}/{{len(segments)}}] уже на Диске. Пропуск.")
        final_audio += AudioSegment.from_wav(out_path)
        final_audio += AudioSegment.silent(duration=250)
        continue

    print(f"Озвучиваю [{{i+1}}/{{len(segments)}}]: {{text}}")

    try:
        tts.tts_to_file(text=text, speaker_wav=speaker_sample, language="en", file_path=out_path)
        final_audio += AudioSegment.from_wav(out_path)
        final_audio += AudioSegment.silent(duration=250)
    except Exception as e:
        print(f"❌ Ошибка на фразе {{i+1}}: {{e}}")

final_output = os.path.join(WORK_DIR, "FINAL_ENGLISH_CLONED.wav")
print("--- ЭТАП 4: Склейка и сохранение ---")
final_audio.export(final_output, format="wav")
print(f"УСПЕХ! Итоговый файл сохранен на твоем Google Диске: {{final_output}}")
"""

with open("run_clone.py", "w", encoding="utf-8") as f:
    f.write(code)

# ==========================================
# ШАГ 4: ЗАПУСК С ВЫВОДОМ В РЕАЛЬНОМ ВРЕМЕНИ
# ==========================================
print("Запускаем процесс клонирования...")
!/usr/local/bin/conda run --no-capture-output -n clone_env python run_clone.py

Подключаем Google Диск для надежного сохранения...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Копируем исходный аудиофайл на Google Диск...
Устанавливаем правильные версии библиотек (с поддержкой видеокарты)...
Запускаем процесс клонирования...
/usr/local/envs/clone_env/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename

>>> ПРОВЕРКА GPU: Видеокарта доступна? False <<<

--- ЭТАП 1: Подготовка эталона ---
Эталон голоса сохранен на Диск.
--- ЭТАП 2: Перевод ---
Переводим аудио в текст...
/usr/local/envs/clone_env/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be 